# [13-3강] 장기 의존성과 경사 소실 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. 반복 tanh 연산의 gradient 확인하기

같은 연산을 여러 번 반복했을 때 시작 Tensor의 gradient가 어떻게 달라지는지 확인합니다.

In [2]:
def grad_after_steps(steps):
    x = torch.tensor([0.5], requires_grad=True)
    h = x
    for _ in range(steps):
        # TODO: h에 tanh 연산을 반복 적용하세요.
        h = torch.tanh(h)
    h.backward()
    return abs(x.grad.item())

print('short:', grad_after_steps(2))
print('long:', grad_after_steps(30))


short: 0.6398079991340637
long: 0.06583694368600845


### 해설 및 실행 결과 해석

- 반복 연산이 길어지면 gradient가 점점 작아질 수 있습니다. 이것이 긴 sequence에서 초기 정보 학습이 어려워지는 경사 소실 문제와 연결됩니다.

## 문제 2. RNN gradient norm 계산하기

RNN으로 sequence 분류 loss를 계산한 뒤 모든 parameter gradient를 하나의 벡터로 보았을 때의 global L2 norm, 즉 `sqrt(sum(grad ** 2))`를 구합니다.

In [7]:
x = torch.randn(8, 20, 3)
y = torch.randint(0, 2, (8,))
rnn = nn.RNN(3, 6, batch_first=True)
head = nn.Linear(6, 2)
loss_fn = nn.CrossEntropyLoss()

out, h = rnn(x)
logits = head(h[-1])
loss = loss_fn(logits, y)
loss.backward()
params = list(rnn.parameters()) + list(head.parameters())
# TODO: backward 후 모든 parameter gradient의 global L2 norm을 계산하세요.
grad_sq_sum = sum(
    p.grad.detach().pow(2).sum().item()
    for p in params
    if p.grad is not None
)
grad_norm = grad_sq_sum ** 0.5
print('loss:', round(loss.item(), 4), 'global_grad_norm:', round(grad_norm, 4))


loss: 0.7227 global_grad_norm: 0.3688


In [8]:
#####
x = torch.randn(8, 20, 3)
y = torch.randint(0, 2, (8,))
rnn = nn.RNN(3, 6, batch_first=True)
head = nn.Linear(6, 2)
loss_fn = nn.CrossEntropyLoss()

out, h = rnn(x)
logits = head(h[-1])
loss = loss_fn(logits, y)
loss.backward()
params = list(rnn.parameters()) + list(head.parameters())
grad_sq_sum = sum(
    p.grad.detach().pow(2).sum().item()
    for p in params
    if p.grad is not None
)
grad_norm = grad_sq_sum ** 0.5
print('loss:', round(loss.item(), 4), 'global_grad_norm:', round(grad_norm, 4))

loss: 0.565 global_grad_norm: 0.405


### 해설 및 실행 결과 해석

- gradient norm은 현재 update 신호의 크기를 보여줍니다. 값이 너무 작으면 학습 신호가 약하고, 너무 크면 학습이 불안정해질 수 있습니다.

## 문제 3. gradient clipping 적용하기

`clip_grad_norm_`이 반환하는 clipping 전 global L2 norm과, clipping 후 gradient에서 다시 계산한 global L2 norm을 비교합니다.

In [9]:
x = torch.randn(8, 40, 3)
y = torch.randint(0, 2, (8,))
rnn = nn.RNN(3, 6, batch_first=True)
head = nn.Linear(6, 2)
loss = nn.CrossEntropyLoss()(head(rnn(x)[1][-1]), y)
loss.backward()
params = list(rnn.parameters()) + list(head.parameters())
max_norm = 1.0
# TODO: clip_grad_norm_을 적용하고 clipping 전·후 global L2 norm을 구하세요.
before_norm = torch.nn.utils.clip_grad_norm_(params, max_norm=max_norm)
after_norm = sum(
    p.grad.detach().pow(2).sum().item()
    for p in params
    if p.grad is not None
)
print('before clipping:', float(before_norm))
print('after clipping:', after_norm)


before clipping: 0.7616448998451233
after clipping: 0.5801029372960329


In [10]:
####
x = torch.randn(8, 40, 3)
y = torch.randint(0, 2, (8,))
rnn = nn.RNN(3, 6, batch_first=True)
head = nn.Linear(6, 2)
loss = nn.CrossEntropyLoss()(head(rnn(x)[1][-1]), y)
loss.backward()
params = list(rnn.parameters()) + list(head.parameters())
max_norm = 1.0
before_norm = torch.nn.utils.clip_grad_norm_(params, max_norm=max_norm)
after_sq_sum = sum(
    p.grad.detach().pow(2).sum().item()
    for p in params
    if p.grad is not None
)
after_norm = after_sq_sum ** 0.5
print('before clipping:', float(before_norm))
print('after clipping:', after_norm)
print('within max_norm:', after_norm <= max_norm + 1e-6)

before clipping: 0.3906809389591217
after clipping: 0.3906809480655054
within max_norm: True


### 해설 및 실행 결과 해석

- clipping은 gradient가 기준보다 클 때 update 크기를 제한합니다. RNN/LSTM처럼 sequence 길이에 영향을 받는 모델에서 학습 안정화에 자주 쓰입니다.